# Wan2.2-Animate — Video Character Swap (replacement / "Mix" mode)

Replace the person in **your video** with the character from **your reference image**, keeping the original motion, expressions, camera movement, lighting and background.

## How to use this notebook
1. **Runtime -> Change runtime type -> GPU -> A100 (80 GB)**. This notebook needs a big GPU; see the check in the first cell.
2. **Runtime -> Run all** (or run the cells top to bottom).
3. When Cell 5 asks, upload **two files**: your driving video and your reference character image. That is the only input the notebook needs from you.
4. Wait. The final cell previews the result and downloads it.

> First run downloads ~52 GB of model weights onto the Colab VM disk (`/content`). It is resumable, but a runtime recycle means downloading again.
> Free/Pro Colab (T4 15 GB, L4 22 GB, A100 40 GB) **cannot** run the full 14B pipeline. Use the GGUF + ComfyUI route on those tiers.


## 1. Preflight — verify the runtime can actually run this

In [ ]:
#@title 1) Preflight check (GPU / RAM / disk)
import os, sys, shutil, torch

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU detected. Runtime > Change runtime type > Hardware accelerator > GPU."

vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
disk = shutil.disk_usage("/content").free / 1024**3
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM: %.1f GB | RAM: %.1f GB | Free disk: %.1f GB" % (vram, ram, disk))

NEED_VRAM, NEED_DISK = 45.0, 65.0
CONTINUE_ANYWAY = False   # set True only if you accept the risk

if vram < NEED_VRAM or disk < NEED_DISK:
    print("STOP: this pipeline needs >= %s GB VRAM and >= %s GB free disk." % (NEED_VRAM, NEED_DISK))
    print("You have %.1f GB VRAM and %.1f GB disk." % (vram, disk))
    print("Switch to an A100 80GB runtime, or use the ComfyUI + GGUF route instead.")
    if not CONTINUE_ANYWAY:
        raise SystemExit("Preflight failed.")
    print("Continuing anyway because CONTINUE_ANYWAY = True.")
else:
    print("Preflight OK.")


## 2. Settings (optional to touch)

In [ ]:
#@title 2) Settings
REPO_DIR = "/content/Wan2.2"
CKPT_DIR = "/content/Wan2.2/Wan2.2-Animate-14B"
WORK_DIR = "/content/work"
IN_VIDEO = f"{WORK_DIR}/driving.mp4"
IN_IMAGE = f"{WORK_DIR}/reference.png"
PROC_DIR = f"{WORK_DIR}/preprocess_replace"
OUT_VIDEO = f"{WORK_DIR}/output/replaced.mp4"

MAX_SECONDS = 10
TARGET_FPS = 30
RES_W, RES_H = 1280, 720

MASK_ITERATIONS, MASK_K = 3, 7
MASK_W_LEN, MASK_H_LEN = 1, 1
SEED = 42

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(os.path.dirname(OUT_VIDEO), exist_ok=True)
print("Work dir:", WORK_DIR)


## 3. Install the official repo and dependencies

In [ ]:
#@title 3) Clone repo + install deps
%cd /content
![ -d Wan2.2 ] || git clone --depth 1 https://github.com/Wan-Video/Wan2.2.git
%cd /content/Wan2.2

!pip -q install -r requirements.txt 2>&1 | tail -5
!pip -q install "huggingface_hub[cli]" hf_transfer iopath hydra-core 2>&1 | tail -3

import torch
if not torch.cuda.is_available():
    print("CUDA lost after install - restoring torch...")
    !pip -q install --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
    import importlib
    importlib.reload(torch)
    assert torch.cuda.is_available(), "Torch still has no CUDA. Restart the runtime and re-run from Cell 3."

import wan
print("wan package OK | torch =", torch.__version__, "| CUDA:", torch.cuda.is_available())


## 4. Download the weights (~52 GB, resumable)

In [ ]:
#@title 4) Download Wan2.2-Animate-14B
%cd /content/Wan2.2
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import snapshot_download

IGNORE = [
    "assets/*",
    "xlm-roberta-large/pytorch_model.bin",
    "xlm-roberta-large/tf_model.h5",
    "xlm-roberta-large/flax_model.msgpack",
    "xlm-roberta-large/onnx/*",
    "process_checkpoint/sam2/sam2_hiera_tiny.pt",
    "process_checkpoint/sam2/sam2_hiera_small.pt",
    "process_checkpoint/sam2/sam2_hiera_base_plus.pt",
]

snapshot_download(
    repo_id="Wan-AI/Wan2.2-Animate-14B",
    local_dir=CKPT_DIR,
    ignore_patterns=IGNORE,
    max_workers=8,
)

from pathlib import Path
need = [
    "models_t5_umt5-xxl-enc-bf16.pth",
    "models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth",
    "Wan2.1_VAE.pth",
    "relighting_lora.ckpt",
    "google/umt5-xxl/tokenizer.json",
    "xlm-roberta-large/sentencepiece.bpe.model",
    "process_checkpoint/det/yolov10m.onnx",
    "process_checkpoint/pose2d/vitpose_h_wholebody.onnx/end2end.onnx",
    "process_checkpoint/sam2/sam2_hiera_large.pt",
]
missing = [p for p in need if not (Path(CKPT_DIR) / p).exists()]
print("Missing:", missing if missing else "none")
assert not missing, "Re-run this cell - the download is resumable and will pick up where it stopped."
print("Weights ready.")


## 5. Upload your video + reference image, then normalise the video

In [ ]:
#@title 5) Upload driving video + reference image
from google.colab import files
import os, cv2
from PIL import Image

print("Select BOTH files at once: your driving video (mp4/mov/avi) and the reference character image.")
uploaded = files.upload()
assert len(uploaded) >= 2, "Please select both the video and the reference image in one go."

VID_EXT = (".mp4", ".mov", ".avi", ".mkv", ".webm")
IMG_EXT = (".png", ".jpg", ".jpeg", ".webp", ".bmp")
vid_src = next((f for f in uploaded if f.lower().endswith(VID_EXT)), None)
img_src = next((f for f in uploaded if f.lower().endswith(IMG_EXT)), None)
assert vid_src, "No video found in your upload."
assert img_src, "No reference image found in your upload."

Image.open(img_src).convert("RGB").save(IN_IMAGE)
w, h = Image.open(IN_IMAGE).size
print("Reference image: %dx%d px" % (w, h))
assert 200 <= w <= 4096 and 200 <= h <= 4096, "Reference image must be 200-4096 px per side."

!ffmpeg -y -loglevel error -i "{vid_src}" -t {MAX_SECONDS} -r {TARGET_FPS} -an -vf "scale='min(1920,iw)':-2,scale=trunc(iw/2)*2:trunc(ih/2)*2" -c:v libx264 -preset veryfast -crf 18 -pix_fmt yuv420p "{IN_VIDEO}"

cap = cv2.VideoCapture(IN_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)
n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
vw = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
vh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print("Driving video: %dx%d, %.0f fps, %d frames (%.1f s)" % (vw, vh, fps, n, n / max(fps, 1)))

if vh > vw:
    RES_W, RES_H = 720, 1280
print("Working resolution area: %dx%d" % (RES_W, RES_H))


## 6. Preprocess — build the pose / face / background / mask materials

In [ ]:
#@title 6) Preprocess (replacement mode)
%cd /content/Wan2.2
from pathlib import Path
from PIL import Image

!python ./wan/modules/animate/preprocess/preprocess_data.py --ckpt_path {CKPT_DIR}/process_checkpoint --video_path "{IN_VIDEO}" --refer_path "{IN_IMAGE}" --save_path "{PROC_DIR}" --resolution_area {RES_W} {RES_H} --fps -1 --iterations {MASK_ITERATIONS} --k {MASK_K} --w_len {MASK_W_LEN} --h_len {MASK_H_LEN} --replace_flag

produced = ["src_pose.mp4", "src_face.mp4", "src_bg.mp4", "src_mask.mp4"]
missing = [f for f in produced if not (Path(PROC_DIR) / f).exists()]
print("Produced:", [f for f in produced if f not in missing] or "NOTHING")

ref = Path(PROC_DIR) / "src_ref.png"
if not ref.exists():
    Image.open(IN_IMAGE).convert("RGB").save(ref)
    print("Wrote missing src_ref.png from your uploaded reference image.")

assert not missing, "Preprocessing failed. The usual cause is a multi-person video - the mask extractor is single-person only."
print("Preprocessing complete.")


## 7. Generate the swapped video (replacement + relighting LoRA)

In [ ]:
#@title 7) Generate (replacement + relighting LoRA)
%cd /content/Wan2.2
import os
SIZE_STR = "1280*720" if RES_W >= RES_H else "720*1280"
if os.path.exists(OUT_VIDEO):
    os.remove(OUT_VIDEO)

!python generate.py --task animate-14B --size {SIZE_STR} --ckpt_dir {CKPT_DIR}/ --src_root_path "{PROC_DIR}/" --refert_num 1 --replace_flag --use_relighting_lora --offload_model True --convert_model_dtype --base_seed {SEED} --save_file "{OUT_VIDEO}"

print("Exists:", os.path.exists(OUT_VIDEO), "| Size (MB):", round(os.path.getsize(OUT_VIDEO)/1024**2, 1) if os.path.exists(OUT_VIDEO) else 0)
assert os.path.exists(OUT_VIDEO), "No output produced - check the log above."


## 8. Preview and download the result

In [ ]:
#@title 8) Preview + download
import os, glob
from IPython.display import Video, display
from google.colab import files

if not os.path.exists(OUT_VIDEO):
    cands = sorted(glob.glob("/content/Wan2.2/*.mp4"), key=os.path.getmtime, reverse=True)
    cands = [c for c in cands if "process" not in c]
    assert cands, "No output video found - check Cell 7's log for an error."
    OUT_VIDEO = cands[0]
    print("Using newest video:", OUT_VIDEO)

PLAYABLE = os.path.splitext(OUT_VIDEO)[0] + "_h264.mp4"
!ffmpeg -y -loglevel error -i "{OUT_VIDEO}" -c:v libx264 -crf 20 -pix_fmt yuv420p "{PLAYABLE}"

display(Video(PLAYABLE, embed=True, width=720))
files.download(PLAYABLE)
print("Download started:", PLAYABLE)
